# Clinical Claims Feature Store

Executable ETL notebook for raw claims validation, feature engineering, SQLite persistence, lineage generation, and sample feature queries.

In [ ]:
from pathlib import Path
import os
import sqlite3
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name != 'feature-store-claims':
    PROJECT_ROOT = Path('/mnt/data/feature-store-claims')

os.environ.setdefault('PROJECT_ROOT', str(PROJECT_ROOT))
os.environ.setdefault('RAW_DATA_PATH', 'data/raw/claims.csv')
os.environ.setdefault('PROCESSED_DIR', 'data/processed')
os.environ.setdefault('OUTPUTS_DIR', 'outputs')
os.environ.setdefault('FEATURE_DB_PATH', 'data/processed/feature_store.db')
os.environ.setdefault('AS_OF_DATE', '2026-06-09')

sys.path.insert(0, str(PROJECT_ROOT / 'src'))

from config import ensure_directories, get_config
from data_loader import prepare_raw_claims
from feature_engineering import (
    build_feature_store,
    build_lineage_table,
    run_duckdb_preview,
    save_outputs,
    write_feature_store,
)
from validator import build_data_quality_metrics, validate_claims_schema

config = get_config()
ensure_directories(config)
config

## 1. Data Loading + Schema Validation

In [ ]:
sample_input = PROJECT_ROOT / 'sample_data' / 'claims.csv'
claims = prepare_raw_claims(sample_input, config.raw_data_path)
validate_claims_schema(claims)
claims.head()

In [ ]:
quality_metrics = build_data_quality_metrics(claims)
quality_metrics

## 2. Feature Calculation with SQL + pandas

In [ ]:
duckdb_preview = run_duckdb_preview(claims)
duckdb_preview[:5]

In [ ]:
features = build_feature_store(claims, config.as_of_date)
lineage = build_lineage_table(features)
features.head(15)

In [ ]:
features.groupby(['entity_type', 'feature_name']).size().reset_index(name='row_count')

## 3. Feature Store Write

In [ ]:
write_feature_store(features, lineage, config.feature_db_path)
save_outputs(claims, features, config.outputs_dir)
print(f'Feature store written to: {config.feature_db_path}')
print(f'Outputs written to: {config.outputs_dir}')

## 4. Sample Queries

In [ ]:
connection = sqlite3.connect(config.feature_db_path)

top_risk_members = pd.read_sql_query(
    """
    SELECT entity_id AS member_id, CAST(feature_value AS REAL) AS risk_score
    FROM features
    WHERE entity_type = 'member' AND feature_name = 'risk_score'
    ORDER BY risk_score DESC
    LIMIT 10;
    """,
    connection,
)
top_risk_members

In [ ]:
quality_leaders = pd.read_sql_query(
    """
    WITH specialty AS (
        SELECT entity_id AS provider_id, feature_value AS specialty
        FROM features
        WHERE entity_type = 'provider' AND feature_name = 'specialty'
    ), quality AS (
        SELECT entity_id AS provider_id, CAST(feature_value AS REAL) AS quality_score_cms
        FROM features
        WHERE entity_type = 'provider' AND feature_name = 'quality_score_cms'
    )
    SELECT specialty.specialty, quality.provider_id, quality.quality_score_cms
    FROM quality
    JOIN specialty USING (provider_id)
    ORDER BY quality.quality_score_cms DESC;
    """,
    connection,
)
quality_leaders

In [ ]:
service_outliers = pd.read_sql_query(
    """
    SELECT entity_id AS service_code
    FROM features
    WHERE entity_type = 'service_line'
      AND feature_name = 'outlier_flag'
      AND feature_value = '1';
    """,
    connection,
)
service_outliers

In [ ]:
lineage_sample = pd.read_sql_query(
    """
    SELECT feature_id, data_source, last_updated
    FROM lineage
    LIMIT 5;
    """,
    connection,
)
connection.close()
lineage_sample